In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv("pathpilot_v1_filtered.csv")
df.head()

,course_id,course_title,url,platform,subject,level,num_subscribers,num_reviews,rating,num_lectures,...,course_type,learning_path,playlist_id,skill_level_score,popularity_score,last_updated,career_relevance_score,combined_features,level_num,title_clean
0,CRS_0001,Python for Everybody Specialization,https://www.coursera.org/specializations/python,Coursera,Python Programming,Beginner,0.184553,0.643222,0.8,96,...,Video,Python Mastery Path,PL_011,1,0.762149,2024,0.537829,Python for Everybody Specialization Python Pro...,1.0,python for everybody specialization
1,CRS_0002,The Complete Python Bootcamp From Zero to Hero,https://www.udemy.com/course/complete-python-b...,Udemy,Python Programming,Beginner,0.276867,0.964854,0.4,155,...,Video,Python Mastery Path,PL_011,1,0.895047,2024,0.597126,The Complete Python Bootcamp From Zero to Hero...,1.0,the complete python bootcamp from zero to hero
2,CRS_0003,Machine Learning Specialization,https://www.coursera.org/specializations/machi...,Coursera,Machine Learning,Beginner,0.138395,0.361794,1.0,95,...,Video,ML Engineer Path,PL_009,1,0.620866,2024,0.463060,Machine Learning Specialization Machine Learni...,1.0,machine learning specialization
3,CRS_0004,Deep Learning Specialization,https://www.coursera.org/specializations/deep-...,Coursera,Deep Learning,Intermediate,0.130702,0.351743,1.0,120,...,Video,ML Engineer Path,PL_009,2,0.609843,2024,0.455650,Deep Learning Specialization Deep Learning Int...,2.0,deep learning specialization
4,CRS_0005,Natural Language Processing Specialization,https://www.coursera.org/specializations/natur...,Coursera,Natural Language Processing,Intermediate,0.061466,0.130621,0.8,120,...,Video,NLP & LLM Engineer Path,PL_010,2,0.367334,2024,0.287635,Natural Language Processing Specialization Nat...,2.0,natural language processing specialization


In [4]:
# df.drop(columns=['carer_relevance_score'], inplace=True)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4859 entries, 0 to 4858
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   course_id               4859 non-null   object 
 1   course_title            4859 non-null   object 
 2   url                     4859 non-null   object 
 3   platform                4859 non-null   object 
 4   subject                 4859 non-null   object 
 5   level                   4859 non-null   object 
 6   num_subscribers         4859 non-null   float64
 7   num_reviews             4859 non-null   float64
 8   rating                  4859 non-null   float64
 9   num_lectures            4859 non-null   int64  
 10  content_duration        4859 non-null   float64
 11  instructor              4859 non-null   object 
 12  language                4859 non-null   object 
 13  tags                    4859 non-null   object 
 14  course_type             4859 non-null   

In [6]:
df.to_csv("D:\Study\Project\PathPilot-AI\data\processed\pathpilot_v1_filtered.csv", index=True)

In [7]:
text_cols = ["course_title", "subject", "level", "tags", "learning_path", "instructor"]

for col in text_cols:
    df[col] = df[col].fillna("")

In [8]:
df["combined_features"] = (
    df["course_title"] + " " +
    df["subject"] + " " +
    df["level"] + " " +
    df["tags"] + " " +
    df["learning_path"] + " " +
    df["instructor"]
)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tdifd = TfidfVectorizer(stop_words="english")
tdifd_matrix = tdifd.fit_transform(df['combined_features'])

In [10]:
tdifd_matrix.shape

(4859, 2356)

In [11]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tdifd_matrix, tdifd_matrix)

In [12]:
cosine_sim.shape

(4859, 4859)

In [13]:
indices = pd.Series(df.index, index=df['course_title']).drop_duplicates()

In [14]:
indices

course_title
Python for Everybody Specialization                                                            0
The Complete Python Bootcamp From Zero to Hero                                                 1
Machine Learning Specialization                                                                2
Deep Learning Specialization                                                                   3
Natural Language Processing Specialization                                                     4
                                                                                            ... 
Business & Product Thinking Topic 14 in Business & Product Thinking – Deep Dive Guide       4854
Business & Product Thinking Business & Product Thinking Topic 14 – Ultimate Course          4855
In-Depth Business & Product Thinking Business & Product Thinking Topic 4                    4856
Business & Product Thinking Topic 2 in Business & Product Thinking – Comprehensive Guide    4857
Mastering Busines

In [15]:
df["title_clean"] = df["course_title"].str.lower().str.strip()

In [16]:
import difflib

def find_best_match(user_input, df):
    user_input = user_input.lower().strip()

    # 1. Exact clean match
    exact_matches = df[df["title_clean"] == user_input]
    if not exact_matches.empty:
        return exact_matches.index[0]

    # 2. Partial contains match
    partial_matches = df[df["title_clean"].str.contains(user_input, na=False)]
    if not partial_matches.empty:
        return partial_matches.index[0]

    # 3. Fuzzy match
    possible_titles = df["title_clean"].tolist()
    close_matches = difflib.get_close_matches(user_input, possible_titles, n=1, cutoff=0.4)

    if close_matches:
        matched_title = close_matches[0]
        return df[df["title_clean"] == matched_title].index[0]

    return None

In [17]:
def recommend_courses(course_title, top_n=5):
    idx = find_best_match(course_title, df)

    if idx is None:
        return "Course not found!"

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    course_indices = [i[0] for i in sim_scores]

    return df[[
        "course_title",
        "subject",
        "level",
        "learning_path",
        "career_relevance_score"
    ]].iloc[course_indices]

In [18]:
recommend_courses("Machine", top_n=5)

,course_title,subject,level,learning_path,career_relevance_score
3935,Ultimate Ensemble Learning for Machine Learning,Machine Learning,Intermediate,ML Engineer Path,0.159008
3902,Comprehensive Machine Learning Logistic Regres...,Machine Learning,All Levels,ML Engineer Path,0.157782
3,Deep Learning Specialization,Deep Learning,Intermediate,ML Engineer Path,0.455650
3924,Mastering Logistic Regression (Machine Learning),Machine Learning,Beginner,ML Engineer Path,0.210948
3919,Masterclass Meta Learning for Machine Learning,Machine Learning,Beginner,ML Engineer Path,0.165892


In [19]:
level_map = {
    "Beginner": 1,
    "Intermediate": 2,
    "Advanced": 3
}

df["level_num"] = df["level"].map(level_map)

In [20]:
def recommend_smart(course_title, top_n=5):
    idx = find_best_match(course_title, df)

    if idx is None:
        return "Course not found!"

    input_level = df.loc[idx, "level_num"]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:20]

    candidate_data = []

    for i, sim_score in sim_scores:
        row = df.iloc[i]

        if row["level_num"] >= input_level and row["level_num"] <= input_level + 1:
            final_score = (
                0.5 * sim_score +
                0.3 * row["career_relevance_score"] +
                0.2 * row["skill_level_score"]
            )

            candidate_data.append({
                "course_title": row["course_title"],
                "subject": row["subject"],
                "level": row["level"],
                "learning_path": row["learning_path"],
                "career_relevance_score": row["career_relevance_score"],
                "skill_level_score": row["skill_level_score"],
                "similarity_score": sim_score,
                "final_score": final_score
            })

    result_df = pd.DataFrame(candidate_data)
    return result_df.sort_values(by="final_score", ascending=False).head(top_n)

In [21]:
recommend_smart("Python for", top_n=5)

,course_title,subject,level,learning_path,career_relevance_score,skill_level_score,similarity_score,final_score
2,Python for Data Structures,Python Programming,Intermediate,Python Mastery Path,0.201899,2,0.499621,0.710380
10,Python for Generators,Python Programming,Intermediate,Python Mastery Path,0.279980,2,0.438637,0.703312
14,Python Networking Bootcamp,Python Programming,Intermediate,Python Mastery Path,0.286590,2,0.434508,0.703231
7,Python Regex in Practice,Python Programming,Intermediate,Python Mastery Path,0.176133,2,0.463407,0.684543
5,Mastering Web APIs (Python Programming),Python Programming,Intermediate,Python Mastery Path,0.053521,2,0.470806,0.651459


In [22]:
df.to_csv("C:/Users/Preneel/Jupyter Notebook/Projects/Path Pilot AI/pathpilot_v1_filtered.csv", index=False)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4859 entries, 0 to 4858
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   course_id               4859 non-null   object 
 1   course_title            4859 non-null   object 
 2   url                     4859 non-null   object 
 3   platform                4859 non-null   object 
 4   subject                 4859 non-null   object 
 5   level                   4859 non-null   object 
 6   num_subscribers         4859 non-null   float64
 7   num_reviews             4859 non-null   float64
 8   rating                  4859 non-null   float64
 9   num_lectures            4859 non-null   int64  
 10  content_duration        4859 non-null   float64
 11  instructor              4859 non-null   object 
 12  language                4859 non-null   object 
 13  tags                    4859 non-null   object 
 14  course_type             4859 non-null   

In [24]:
user_profile = {
    "goal": "ai_career",        # job / freelancing / ai / web_dev
    "level": "beginner",       # beginner / intermediate / advanced
    "preferred_duration": "short",  # short / medium / long
    "language": "English",     # optional
}

In [25]:
def duration_category(hours):
    if hours < 2:
        return "short"
    elif hours < 10:
        return "medium"
    else:
        return "long"

df["duration_category"] = df["content_duration"].apply(duration_category)

In [26]:
level_map = {
    "Beginner": 1,
    "Intermediate": 2,
    "Advanced": 3
}

df["level_num"] = df["level"].map(level_map)

In [30]:
def goal_tags(row):
    tags = []

    if row["level"].lower() == "beginner":
        tags.append("foundation")

    if row["level"].lower() in ["intermediate", "advanced"]:
        tags.append("skill_upgrade")

    if row["career_relevance_score"] > 0.7:
        tags.append("job_ready")

    if "web" in row["subject"].lower():
        tags.append("freelancing")

    if "machine learning" in row["course_title"].lower():
        tags.append("ai_career")

    return ",".join(tags)

df["goal_alignment_tags"] = df.apply(goal_tags, axis=1)

In [33]:
df["goal_alignment_tags"]

0                 foundation
1                 foundation
2       foundation,ai_career
3              skill_upgrade
4              skill_upgrade
                ...         
4854              foundation
4855           skill_upgrade
4856           skill_upgrade
4857           skill_upgrade
4858           skill_upgrade
Name: goal_alignment_tags, Length: 4859, dtype: object

In [32]:
df["goal_alignment_tags"] = df["goal_alignment_tags"].str.lower()

In [42]:
filtered_df = df.copy()

In [43]:
def recommend_for_user(user_profile, top_n=5):
    
    filtered_df = df.copy()

    # 1. Filter by level (same or next level)
    user_level_num = level_map[user_profile["level"].capitalize()]
    
    filtered_df = filtered_df[
        (filtered_df["level_num"] >= user_level_num) &
        (filtered_df["level_num"] <= user_level_num + 1)
    ]

    # 2. Filter by duration
    filtered_df = filtered_df[
        filtered_df["duration_category"] == user_profile["preferred_duration"]
    ]

    # 3. Filter by language (optional)
    if "language" in user_profile:
        filtered_df = filtered_df[
            filtered_df["language"].str.lower() == user_profile["language"].lower()
        ]

    # 4. Compute Goal Match Score
    def goal_match(tags):
        if user_profile["goal"] in tags:
            return 1
        return 0

    filtered_df["goal_match"] = filtered_df["goal_alignment_tags"].apply(goal_match)

    # 5. Final Score
    filtered_df["final_score"] = (
        0.4 * filtered_df["goal_match"] +
        0.3 * filtered_df["career_relevance_score"] +
        0.2 * filtered_df["popularity_score"] +
        0.1 * filtered_df["skill_level_score"]
    )

    # 6. Sort & Return
    return filtered_df.sort_values(by="final_score", ascending=False)[[
        "course_title",
        "subject",
        "level",
        "learning_path",
        "goal_alignment_tags",
        "final_score"
    ]].head(top_n)

In [44]:
user_profile = {
    "goal": "ML",
    "level": "beginner",
    "preferred_duration": "medium",
    "language": "English"
}

recommend_for_user(user_profile, top_n=5)

,course_title,subject,level,learning_path,goal_alignment_tags,final_score
7,Data Structures Easy to Advanced - Full Course,Data Structures & Algorithms,Beginner,Software Engineer Path,"foundation,job_ready",0.516985
2101,From Scratch JavaScript: Testing,JavaScript,Beginner,Full-Stack Developer Path,foundation,0.507672
2176,Modern JavaScript: TypeScript Basics,JavaScript,Beginner,Full-Stack Developer Path,foundation,0.487036
1619,Complete Divide & Conquer Course,Data Structures & Algorithms,Beginner,Software Engineer Path,foundation,0.482659
2032,Complete Web Web Components Course,Web Development,Beginner,Full-Stack Developer Path,"foundation,freelancing",0.457215


In [46]:
filtered_df = filtered_df[
    filtered_df["subject"].str.lower().str.contains(user_profile["goal"])
]